# 策略概述

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

**Agglomerative SEC-PIT Beta** 在 Agglomerative（價格 PCA ⊕ 公司基本面）分群距離中，加入 **Beta 系統性風險先驗**——與 FMP 版（`#12`）**唯一差異是多一個 Beta 特徵區塊**，藉此隔離「基本面風險先驗」對分群→配對品質的**淨貢獻**（研究框架 #5）：

1. 借用 `HDBSCAN_PCA_Loadings` 萃取**價格 PCA 因子載荷**
2. SEC／FMP **Point-in-Time** parquet 以 asof 對齊 `form_end` 取市值／PE（無前視）
3. **Beta** = 形成窗報酬對橫斷面市場因子回歸斜率（全覆蓋、天生 PIT）
4. 各特徵區塊分別標準化＋加權組合 → **Agglomerative**（average linkage）分群
5. 群內 **SSD Rolling** 排序，取前 `top_n`
6. 交易期 Z-Score 路徑 B



## 為何只加 Beta，不加槓桿／獲利率

**Beta（系統性風險）相近**的兩檔股票，其 spread 更**市場中性**、更易均值回歸，是理論上最貼合市場中性配對的基本面風險特徵；且 Beta 由形成窗價格回歸即得——**全覆蓋、天生 point-in-time、無前視**。

相對地，SEC EDGAR XBRL 的基本面覆蓋率在本資料集極低：市值／PE 全期僅約 10%、2009 年前近乎 0（XBRL 制度 2009 才普及）。若再從 XBRL 拉槓桿（D/E）、獲利率（ROE／margin），同樣 0%（2009 前）＋稀疏（之後約 15%）→ **90%+ 需群組中位數插補 = 引入雜訊而非訊號**。故只加 Beta，市值／PE 沿用 PIT 有資料處、缺漏以產業中位數插補。

::: {.callout-important}

**實測結論**：等權（`beta_feature_weight=1.0`）加入 Beta **反而全面變差**——高變異的 Beta 主導了分群距離，把 β 相近但行為／產業不同的股票硬拉在一起，群質下降。與 #2/#3/#4 同一母題：**好的基礎表徵已捕捉結構，額外堆疊特徵常是稀釋而非增益**。`beta_feature_weight` 已納入敏感性掃描（$\{0,0.25,0.5,1.0\}$，$0=$ 無 β 控制組 $\approx$ FMP）。詳見績效比較報告。

:::


# 參考文獻與引用對應


## 文獻 1：Hong & Hwang (2021)

> Hong, S., & Hwang, S. (2021). In search of pairs using firm fundamentals: Is pairs trading profitable? *The European Journal of Finance*.

以公司基本面（規模、估值等）尋找配對——本策略「以基本面特徵輔助分群」的直接依據。


## 文獻 2：Sharpe (1964)

> Sharpe, W. F. (1964). Capital asset prices: A theory of market equilibrium under conditions of risk. *The Journal of Finance*, **19**(3), 425–442.

市場模型 $r_i=\alpha_i+\beta_i r_m+\varepsilon_i$ 的 $\beta_i$ 度量**系統性風險**——本策略「階段 3」以 Beta 相近作為市場中性配對的風險先驗。


## 文獻 3：Ward (1963)／Avellaneda & Lee (2010)

> Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301), 236–244.

> Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7), 761–782.

前者為 Agglomerative 階層分群（「階段 4」）依據；後者為報酬 PCA 因子載荷（「階段 1」）依據。


## 文獻 4：Engle & Granger (1987)／Krauss, Do & Huck (2016)

> Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction. *Econometrica*, **55**(2), 251–276.

> Krauss, C., Do, X. A., & Huck, N. (2016). The profitability of pairs trading strategies: Distance, cointegration and copula methods. *Quantitative Finance*, **17**(3), 469–488.

「階段 5」群內 min-SSD 排序 + ADF／半衰期／Hurst 過濾（沿用 SSD Rolling 排序流程）依據。


# 各階段行為


## 階段 1：價格 PCA 因子載荷（依據：Avellaneda & Lee 2010）

借用 `HDBSCAN_PCA_Loadings._build_feature_matrix()`：對形成窗標準化日報酬做 PCA，取前 `pca_n_components`（=5）主成分的因子載荷（以 $\sqrt{\text{特徵值}}$ 加權）作為**價格行為**座標。


## 階段 2：SEC／FMP PIT 基本面 asof 對齊

讀 Point-in-Time 基本面資料（MultiIndex `date, ticker`），取 $\le$ `form_end` 的最近月份快照，逐股取 $\ln(1+\text{市值})$ 與盈餘殖利率 $1/\text{PE}$。**Point-in-Time 對齊確保無前視**；缺漏以正規化產業中位數插補（`_impute_by_group`）+ winsorize。


## 階段 3：Beta 風險先驗（依據：Sharpe 1964）

對每檔股票，以形成窗日報酬對**橫斷面平均報酬（市場因子）**做單因子回歸取斜率：

$$\beta_i=\frac{\operatorname{Cov}(r_i,\,r_{mkt})}{\operatorname{Var}(r_{mkt})},\qquad r_{mkt,t}=\frac{1}{N}\sum_j r_{j,t}$$

**全覆蓋、天生 PIT、無前視**（僅用形成窗資料）。由 `_rolling_betas()` 計算，獨立標準化後以 `beta_feature_weight` 加權，成為特徵矩陣中獨立一欄。


## 階段 4：特徵區塊加權組合 + Agglomerative 分群（依據：Ward 1963）

各區塊**分別**標準化再依權重組合（避免單一 joint scaler 讓 one-hot 稀釋距離）：

$$X=[\,w_p\,\text{PCA}\;\|\;w_f\,(\ln\text{Cap},\,1/\text{PE})\;\|\;w_\beta\,\beta\;\|\;w_s\,\text{GICS one-hot}\,]$$

AgglomerativeClustering（average linkage、依合併距離分位數 `agg_threshold_percentile`=75 校準 `distance_threshold`）分群；過小群（< `min_cluster_size`=5）併入 "Unknown" 排除。


## 階段 5：群內 SSD Rolling 排序（依據：GGR 2006／Krauss et al. 2016）

將分群標籤當作 `sector_mapping` 餵給 `ssd_rolling.Formation`，完整複用其 min-SSD 排序 + Engle-Granger ADF + 半衰期 + Hurst 篩選流程，取前 `top_n`。回填真實 GICS 產業、群集 ID、市值、PE 供報表分析。


## 階段 6：交易期參數使用

交易期於標準化空間重建 spread（Z-Score 路徑 B），輸出 `Ticker_A/B, Rank, Hedge_Ratio, Spread_Mean/Std, Log_Mean/Std_A/B` 供 Z-Score 交易模組 使用；DRL 疊加版（若有）借用同配對改走 DRL 門檻選擇模組。


# 參數總表

| 參數 | 值 | 說明 |
| :--- | :---: | :--- |
| 形成窗 $F$ / 滾動步長 | 252 / 21 交易日 | — |
| `top_n` | 網格 [1, 3, 5, 10, 20] | 每期配對數 |
| `pca_n_components` | 5 | 價格報酬 PCA 因子數 |
| `beta_feature_weight` | **1.0**（敏感性掃 $\{0,0.25,0.5,1.0\}$） | Beta 風險先驗權重 |
| `price/fundamentals/sector_onehot_weight` | 1.0 / 1.0 / 1.0 | 各特徵區塊權重 |
| `agg_linkage` / `agg_threshold_percentile` | average / 75 | Agglomerative 分群 |
| `min_cluster_size` | 5 | 過小群併入 Unknown |
| `adf_pvalue_threshold` | 0.05 | 群內 ADF 顯著水準 |
| 基本面來源 | SEC/FMP PIT parquet | asof 對齊 `form_end`，無前視 |
